# Submissão — Opção B: Fine-tuning DeBERTa-v3-base

**Treino:** dataset próprio do HuggingFace (`BrunoFilipeRDS/50k-balanced-5-classes`)
**Modelo:** `microsoft/deberta-v3-base`  


In [1]:
import os, re
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from torch.optim import AdamW
from datasets import load_dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ─── Ficheiros ────────────────────────────────────────────────────────────────
EXAMPLE_CSV = 'dataset-exemplos.csv'  # só para avaliar
PREDICT_CSV = 'subm2.csv'
OUTPUT_CSV  = 'subm2-g5-MECD-B_base.csv'
MODEL_DIR   = 'deberta_base_finetuned_base'
MODEL_NAME  = 'microsoft/deberta-v3-base'

# ─── Hiperparâmetros ──────────────────────────────────────────────────────────
MAX_LEN      = 128
BATCH_SIZE   = 4       # obrigatório para caber nos 6GB VRAM
EPOCHS       = 8
LR           = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
VAL_SIZE     = 0.1
SEED         = 8088135
PER_CLASS    = 10000   # todos os 50k (10k por classe)

CLASSES   = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']
LABEL2IDX = {c: i for i, c in enumerate(CLASSES)}
IDX2LABEL = {i: c for i, c in enumerate(CLASSES)}

LABEL_NORM = {
    'anthropic': 'Anthropic', 'claude'  : 'Anthropic',
    'google'   : 'Google',    'gemini'  : 'Google',    'gemma'  : 'Google',
    'human'    : 'Human',
    'meta'     : 'Meta',      'llama'   : 'Meta',
    'openai'   : 'OpenAI',    'gpt'     : 'OpenAI',    'chatgpt': 'OpenAI',
}

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Config OK.')

Device: cuda
Config OK.


In [2]:
# ─── Carregar e preparar dataset ──────────────────────────────────────────────
def clean_text(text):
    text = str(text)
    text = re.sub(r'<.*?>', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

print('A carregar dataset do HuggingFace...')
df = load_dataset('BrunoFilipeRDS/50k-balanced-5-classes', split='train').to_pandas()
df.columns = df.columns.str.strip()

df = df.rename(columns={'text': 'Text', 'label': 'Labels'})
df['Text']   = df['Text'].astype(str).apply(clean_text)
df['Labels'] = df['Labels'].astype(str).str.strip().str.lower().map(
    lambda x: LABEL_NORM.get(x, x)
)

df = df[df['Labels'].isin(CLASSES)][['Text', 'Labels']].reset_index(drop=True)
print(f'Exemplos válidos: {len(df)}')
print(df['Labels'].value_counts().to_string())


frames = []
for label, group in df.groupby('Labels'):
    frames.append(group.sample(min(len(group), PER_CLASS), random_state=SEED))

df = (
    pd.concat(frames, ignore_index=True)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)
print(f'\nApós undersample ({PER_CLASS}/classe): {len(df)} exemplos')
print(df['Labels'].value_counts().to_string())

A carregar dataset do HuggingFace...
Exemplos válidos: 250000
Labels
OpenAI       50000
Google       50000
Meta         50000
Anthropic    50000
Human        50000

Após undersample (10000/classe): 50000 exemplos
Labels
Anthropic    10000
Google       10000
Human        10000
OpenAI       10000
Meta         10000


In [3]:
# ─── Dataset PyTorch + DataLoaders ────────────────────────────────────────────
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.enc = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt',
        )
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids'     : self.enc['input_ids'][idx],
            'attention_mask': self.enc['attention_mask'][idx],
            'labels'        : self.labels[idx],
        }

print(f'A carregar tokenizer {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

texts  = df['Text'].tolist()
labels = df['Labels'].map(LABEL2IDX).tolist()

X_tr, X_val, y_tr, y_val = train_test_split(
    texts, labels, test_size=VAL_SIZE, stratify=labels, random_state=SEED
)
print(f'Treino: {len(X_tr)} | Validação: {len(X_val)}')

train_loader = DataLoader(TextDataset(X_tr,  y_tr,  tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TextDataset(X_val, y_val, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)
print('DataLoaders prontos.')

A carregar tokenizer microsoft/deberta-v3-base...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

c:\Users\bruno\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bruno\.cache\huggingface\hub\models--microsoft--deberta-v3-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Treino: 45000 | Validação: 5000
DataLoaders prontos.


In [4]:
# ─── Modelo + Otimizador ─────────────────────────────────────────────────────
print(f'A carregar {MODEL_NAME}...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(CLASSES), id2label=IDX2LABEL, label2id=LABEL2IDX, torch_dtype=torch.float32,
).to(DEVICE)
print(f'Parâmetros: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

no_decay  = ['bias', 'LayerNorm.weight']
optimizer = AdamW([
    {'params': [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': WEIGHT_DECAY},
    {'params': [p for n,p in model.named_parameters() if     any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
], lr=LR)

total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
print(f'Steps totais: {total_steps}')

A carregar microsoft/deberta-v3-base...


`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias        

Parâmetros: 184,425,989
Steps totais: 90000


In [5]:
# ─── Treino com Early Stopping ────────────────────────────────────────────────
def evaluate(m, loader):
    m.eval()
    loss_s, correct, total, preds_a, labs_a = 0, 0, 0, [], []
    with torch.no_grad():
        for b in loader:
            out = m(
                input_ids      = b['input_ids'].to(DEVICE),
                attention_mask = b['attention_mask'].to(DEVICE),
                labels         = b['labels'].to(DEVICE),
            )
            loss_s  += out.loss.item()
            p        = out.logits.argmax(-1)
            correct += (p == b['labels'].to(DEVICE)).sum().item()
            total   += len(b['labels'])
            preds_a.extend(p.cpu().tolist())
            labs_a.extend(b['labels'].tolist())
    return loss_s / len(loader), correct / total, preds_a, labs_a

os.makedirs(MODEL_DIR, exist_ok=True)
best_acc, patience, pat_cnt = 0, 3, 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0
    for b in train_loader:
        optimizer.zero_grad()
        out = model(
            input_ids      = b['input_ids'].to(DEVICE),
            attention_mask = b['attention_mask'].to(DEVICE),
            labels         = b['labels'].to(DEVICE),
        )
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        tr_loss += out.loss.item()

    vl, va, _, _ = evaluate(model, val_loader)
    print(f'Epoch {epoch}/{EPOCHS} | Train Loss: {tr_loss/len(train_loader):.4f} | Val Loss: {vl:.4f} | Val Acc: {va:.4f}')

    if va > best_acc:
        best_acc, pat_cnt = va, 0
        model.save_pretrained(MODEL_DIR)
        tokenizer.save_pretrained(MODEL_DIR)
        print(f'  → Guardado (acc={va:.4f})')
    else:
        pat_cnt += 1
        if pat_cnt >= patience:
            print(f'Early stopping ao epoch {epoch}')
            break

print(f'\nMelhor Val Acc (dataset próprio): {best_acc:.4f}')

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Epoch 1/8 | Train Loss: 0.7797 | Val Loss: 0.6206 | Val Acc: 0.8314


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Guardado (acc=0.8314)
Epoch 2/8 | Train Loss: 0.4875 | Val Loss: 0.4495 | Val Acc: 0.9076


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

  → Guardado (acc=0.9200)
Epoch 7/8 | Train Loss: 0.0627 | Val Loss: 0.6917 | Val Acc: 0.9204


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Guardado (acc=0.9204)
Epoch 8/8 | Train Loss: 0.0375 | Val Loss: 0.7160 | Val Acc: 0.9220


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Guardado (acc=0.9220)

Melhor Val Acc (dataset próprio): 0.9220


In [6]:
# ─── Relatório no set de validação (dataset próprio) ─────────────────────────
m_best = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
_, _, pv, lv = evaluate(m_best, val_loader)
print('=== Validação no dataset próprio ===')
print(classification_report(lv, pv, target_names=CLASSES, zero_division=0))

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

=== Validação no dataset próprio ===
              precision    recall  f1-score   support

   Anthropic       0.96      0.89      0.92      1000
      Google       0.97      0.92      0.95      1000
       Human       1.00      0.99      0.99      1000
        Meta       0.83      0.97      0.90      1000
      OpenAI       0.87      0.84      0.86      1000

    accuracy                           0.92      5000
   macro avg       0.93      0.92      0.92      5000
weighted avg       0.93      0.92      0.92      5000



---
Avaliar no dataset de exemplo


In [7]:
try:
    df_ex = pd.read_csv(EXAMPLE_CSV, sep=';', encoding='utf-8-sig')
    df_ex.columns = df_ex.columns.str.strip()
    df_ex['Text']   = df_ex['Text'].astype(str).apply(clean_text)
    df_ex['Label'] = df_ex['Label'].str.strip()
    df_ex = df_ex[df_ex['Label'].isin(CLASSES)].reset_index(drop=True)
    print(f'Exemplos do professor carregados: {len(df_ex)}')

    tok_ex = AutoTokenizer.from_pretrained(MODEL_DIR)
    m_ex   = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
    m_ex.eval()

    y_true_ex, y_pred_ex = [], []
    for i in range(0, len(df_ex), BATCH_SIZE):
        batch_texts  = df_ex['Text'].iloc[i:i+BATCH_SIZE].tolist()
        batch_labels = df_ex['Label'].iloc[i:i+BATCH_SIZE].map(LABEL2IDX).tolist()
        enc = tok_ex(
            batch_texts, truncation=True, padding='max_length',
            max_length=MAX_LEN, return_tensors='pt',
        ).to(DEVICE)
        with torch.no_grad():
            preds = m_ex(**enc).logits.argmax(-1).cpu().tolist()
        y_pred_ex.extend(preds)
        y_true_ex.extend(batch_labels)

    acc_ex = accuracy_score(y_true_ex, y_pred_ex)
    print(f'\n=== Avaliação no dataset de exemplo do professor ===')
    print(f'Accuracy: {acc_ex:.4f}  ({acc_ex*100:.1f}%)')
    print(classification_report(y_true_ex, y_pred_ex, target_names=CLASSES, zero_division=0))

except FileNotFoundError:
    print(f'Ficheiro {EXAMPLE_CSV} não encontrado — salta se não tiveres o dataset com texto.')

Exemplos do professor carregados: 125


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


=== Avaliação no dataset de exemplo do professor ===
Accuracy: 0.4880  (48.8%)
              precision    recall  f1-score   support

   Anthropic       0.56      0.39      0.46        23
      Google       0.52      0.81      0.63        16
       Human       0.85      0.33      0.47        52
        Meta       0.63      0.71      0.67        17
      OpenAI       0.22      0.59      0.32        17

    accuracy                           0.49       125
   macro avg       0.56      0.56      0.51       125
weighted avg       0.64      0.49      0.50       125



---
## Previsão do dataset do professor (`subm2.csv`)

In [ ]:
tok_inf = AutoTokenizer.from_pretrained(MODEL_DIR)
m_inf   = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
m_inf.eval()

df_pred = pd.read_csv(PREDICT_CSV, sep=';', encoding='utf-8-sig')
df_pred.columns = df_pred.columns.str.strip()
df_pred['Text'] = df_pred['Text'].astype(str).apply(clean_text)
print(f'Registos a prever: {len(df_pred)}')

all_preds = []
for i in range(0, len(df_pred), BATCH_SIZE):
    enc = tok_inf(
        df_pred['Text'].iloc[i:i+BATCH_SIZE].tolist(),
        truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt',
    ).to(DEVICE)
    with torch.no_grad():
        all_preds.extend(m_inf(**enc).logits.argmax(-1).cpu().tolist())

df_out = pd.DataFrame({'ID': df_pred['ID'], 'Labels': [IDX2LABEL[i] for i in all_preds]})
df_out.to_csv(OUTPUT_CSV, index=False, sep=';')
print(f'\nGuardado: {OUTPUT_CSV}')
print(df_out['Labels'].value_counts().to_string())
print(df_out.head(10).to_string(index=False))

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

FileNotFoundError: [Errno 2] No such file or directory: 'subm2.csv'

: 